# 02 · Contact measures and the day-730 landmark

Builds the two contact measures (longest gap in recorded contact, and unique contact days per quarter for the trajectory models), the index-episode features, and the analysis samples for the association analysis.

**Reads (from notebook 01, in `DATA_DIR`):** `eligibility`, `outcomes`

**Produces (patient-level, `DATA_DIR`):** `gaps`, `quarterly_unique`, `index_episode`, `landmark_build`

**Produces (aggregate, `results/`):** `02_landmark_counts_<mode>.json`

**Paper items:** Methods, *Landmark and follow-up* and *Contact measures* (the gap measure G(i) and the trajectory input U(i,q)); Figure 1A (association branch); Table 4 rows by gap; Table S7 gap rows; Table S9 association rows; gap spline knots (Supplementary Methods).

| Sample | Rule | Paper |
|---|---|---|
| Primary landmark population | Alive at day 730, confirmed by any record after day 730; no cardiovascular hospitalization before index or on days 1–730; follow-up beyond day 730 | 14,776 (645 events) |
| Day-730-information sample | No recorded death by day 730 and hospital/ED contact on days 366–730; same event and follow-up rules | 7,374 (312 events) |
| Events before index kept | Primary rules, but a cardiovascular hospitalization before index is allowed | 15,962 (775 events) |

## 0 · Setup

### About `REPLICATE_OCTOBER`

The October 2026 analysis computed day-730 contact (any record after day 730, hospital/ED contact on days 366–730, last hospital/ED contact) **only for patients who passed two older eligibility rules** (≥ 2 BP dates; ≥ 2 encounters or antihypertensive orders). When those rules were dropped, the other 5,328 at-index patients were added with these fields blank, so none of them could be confirmed alive at day 730.

- `REPLICATE_OCTOBER = True` blanks those fields for the 5,328 patients and should reproduce the Draft 7 numbers (14,776 patients, 645 events).
- `REPLICATE_OCTOBER = False` (default) uses everyone's actual contact. This is the corrected analysis.

Run the notebook once each way to compare. Outputs are saved with the mode in the file name, so the two runs don't overwrite each other.

In [ ]:
# ---- edit these ----
REPO_DIR   = "/content/drive/MyDrive/Ronveer Sandhu/Project 1 -MIMIC analysis-hypertension retention/htn-bp-followup-mimic"
MIMIC_HOSP = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/hosp"
MIMIC_ED   = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/ed"
DATA_DIR   = "/content/drive/MyDrive/htn_bp_data"

REPLICATE_OCTOBER = False     # True = reproduce Draft 7 exactly; False = corrected

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install duckdb==1.3.2

import os, sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from htn_utils import *

import numpy as np
import pandas as pd

RESULTS_DIR = os.path.join(REPO_DIR, "results")
MODE = "october" if REPLICATE_OCTOBER else "corrected"
print("mode:", MODE)

el = load_df(DATA_DIR, "eligibility")
oc = load_df(DATA_DIR, "outcomes")
el["index_date"] = pd.to_datetime(el.index_date)

## 1 · Day-730 survival status

Recomputed from the eligibility table. With `REPLICATE_OCTOBER = True`, contact fields are blanked for at-index patients who fail the old rules, as in the October code.

In [ ]:
contact_cols = ["last_hosp_day", "last_any_day", "hosp_366_730", "any_after_730", "capture_end_day"]
failed_old_rules = el.at_index_cohort & ~(el.bp_rule & el.enc_rule)

if REPLICATE_OCTOBER:
    el.loc[failed_old_rules, contact_cols] = np.nan
    print(f"blanked day-730 contact for {int(failed_old_rules.sum()):,} at-index patients who fail the old rules")

el["status730"] = status730(el)

at = el[el.at_index_cohort].copy()
check("At-index cohort", len(at), 37183)
print("\nSurvival status at day 730:")
print(at.status730.value_counts().to_string())
if REPLICATE_OCTOBER:
    check("Survival not confirmed (Figure 1A)", int((at.status730 == "survival unknown").sum()), 14903)

## 2 · Recorded contact, days 0–730

Same encounter and BP definitions as notebook 01, step 2, plus antihypertensive prescription orders. One row per patient, day, and source.

In [ ]:
con = connect(MIMIC_HOSP, MIMIC_ED)

con.execute('''
CREATE OR REPLACE TABLE encounters AS
WITH adm AS (SELECT subject_id, CAST(admittime AS DATE) AS enc_date, 1 AS rank FROM admissions),
     edv AS (SELECT subject_id, CAST(intime AS DATE) AS enc_date, 2 AS rank
             FROM ed_edstays WHERE hadm_id IS NULL),
     opt AS (SELECT DISTINCT subject_id, CAST(chartdate AS DATE) AS enc_date, 3 AS rank FROM omr),
     u   AS (SELECT * FROM adm UNION ALL SELECT * FROM edv UNION ALL SELECT * FROM opt)
SELECT subject_id, enc_date,
       CASE MIN(rank) WHEN 1 THEN 'inpatient' WHEN 2 THEN 'ed' ELSE 'outpatient' END AS enc_type
FROM u GROUP BY subject_id, enc_date''')

con.execute('''
CREATE OR REPLACE TABLE bp AS
SELECT * FROM (
  SELECT subject_id, CAST(chartdate AS DATE) AS bp_date,
         TRY_CAST(SPLIT_PART(result_value, '/', 1) AS INTEGER) AS sbp,
         TRY_CAST(SPLIT_PART(result_value, '/', 2) AS INTEGER) AS dbp
  FROM omr WHERE result_name LIKE 'Blood Pressure%'
) WHERE sbp BETWEEN 60 AND 300 AND dbp BETWEEN 30 AND 200''')

ids = el[["subject_id", "index_date"]].copy()
ids["index_date"] = ids.index_date.dt.date
con.register("cohort_ids", ids)

ct = con.execute(f'''
WITH e AS (SELECT subject_id, CAST(enc_date AS DATE) AS d, enc_type AS src FROM encounters),
     b AS (SELECT subject_id, bp_date AS d, 'bp' AS src FROM bp),
     r AS (SELECT subject_id, CAST(starttime AS DATE) AS d, 'rx' AS src FROM prescriptions
           WHERE {sql_contains('drug', ANTIHTN)}),
     u AS (SELECT * FROM e UNION ALL SELECT * FROM b UNION ALL SELECT * FROM r)
SELECT u.subject_id, DATE_DIFF('day', CAST(c.index_date AS DATE), u.d) AS dd, u.src
FROM u JOIN cohort_ids c ON c.subject_id = u.subject_id
WHERE u.d >= CAST(c.index_date AS DATE)
  AND DATE_DIFF('day', CAST(c.index_date AS DATE), u.d) <= 730''').df()

print(f"contact rows: {len(ct):,} | patients: {ct.subject_id.nunique():,}")
print(ct.src.value_counts().to_string())

## 3 · Longest gap in recorded contact

With d(1) < … < d(m) the days from 0 to 730 with a **blood pressure reading or an admission** (the index reading is day 0):

G(i) = max{ d(2) − d(1), …, d(m) − d(m−1), 730 − d(m) }

The last term truncates the final interval at the landmark, so a gap does not require a later return. `gapped` = G(i) > 365.

In [ ]:
def longest_gap(df_, ids_all):
    out = {}
    for sid, s in df_.groupby("subject_id").dd:
        d = np.unique(np.r_[0, s.values])          # day 0 is always a contact (index BP)
        g = np.diff(d).max() if len(d) > 1 else 0
        out[sid] = max(g, LANDMARK_DAY - d[-1])
    # patients with no BP or admission after index: one interval of 730 days
    return pd.Series(out).reindex(ids_all).fillna(LANDMARK_DAY)

bp_or_adm = ct[ct.src.isin(["bp", "inpatient"])]
gaps = longest_gap(bp_or_adm, el.subject_id.values).rename("longest_gap").rename_axis("subject_id").reset_index()
gaps["gapped"] = (gaps.longest_gap > 365).astype(int)
for t in (180, 270):
    gaps[f"gap_gt_{t}"] = (gaps.longest_gap > t).astype(int)
print(gaps.longest_gap.describe(percentiles=[.25, .5, .75]).round(0).to_dict())

## 4 · Trajectory input: unique contact days per quarter

U(i,q) = number of **unique days** in 91-day quarter q with any recorded encounter, blood pressure reading, or antihypertensive prescription order. A day with several kinds of contact counts once. Quarters 1–7 (days 91–727) close before the landmark. Notebook 05 fits the trajectory models to log(1 + U).

In [ ]:
q17 = ct[ct.dd.between(91, 727)].copy()
q17["q"] = q17.dd // 91
uniq = (q17.drop_duplicates(["subject_id", "dd"])
            .groupby(["subject_id", "q"]).size().rename("n_unique").reset_index())

grid = pd.MultiIndex.from_product([el.subject_id.values, range(1, 8)], names=["subject_id", "q"]).to_frame(index=False)
quarterly = grid.merge(uniq, on=["subject_id", "q"], how="left").fillna({"n_unique": 0})
quarterly["n_unique"] = quarterly.n_unique.astype(int)
print(f"quarterly rows: {len(quarterly):,} (patients x 7 quarters)")
print("mean unique contact days per quarter:", quarterly.groupby("q").n_unique.mean().round(2).to_dict())

## 5 · Index-episode features

Describe the contact that started the cohort: whether an admission began on the index date (emergency, urgent, observation, or elective), emergency department arrival, length of stay, and discharge destination. Patients without an admission on the index date are labelled `outpatient`.

In [ ]:
adm = con.execute('''SELECT subject_id, hadm_id, admittime, dischtime, admission_type,
                            discharge_location, edregtime FROM admissions''').df()
adm["admittime"] = pd.to_datetime(adm.admittime)
adm["dischtime"] = pd.to_datetime(adm.dischtime)
adm = adm.merge(el[["subject_id", "index_date"]], on="subject_id")
adm["admday"] = (adm.admittime.dt.normalize() - adm.index_date).dt.days
idxadm = adm[adm.admday == 0].sort_values("admittime").drop_duplicates("subject_id")

def adm_grp(t):
    if t in ("EW EMER.", "DIRECT EMER."): return "emergency"
    if t == "URGENT": return "urgent"
    if t in ("EU OBSERVATION", "OBSERVATION ADMIT", "DIRECT OBSERVATION"): return "observation"
    return "elective"

def disch_grp(x):
    if pd.isna(x): return "none"
    x = str(x).upper()
    return "home" if "HOME" in x else ("snf_rehab" if ("SKILLED" in x or "REHAB" in x) else "other")

idxadm = idxadm.assign(idx_adm_type=idxadm.admission_type.map(adm_grp),
                       idx_los=(idxadm.dischtime - idxadm.admittime).dt.total_seconds() / 86400,
                       idx_ed=idxadm.edregtime.notna().astype(int),
                       idx_disch=idxadm.discharge_location.map(disch_grp))

index_episode = el[["subject_id"]].merge(
    idxadm[["subject_id", "idx_adm_type", "idx_los", "idx_ed", "idx_disch"]], on="subject_id", how="left")
index_episode["has_idx_adm"]  = index_episode.idx_adm_type.notna().astype(int)
index_episode["idx_adm_type"] = index_episode.idx_adm_type.fillna("outpatient")
index_episode["idx_disch"]    = index_episode.idx_disch.fillna("outpatient")
index_episode["idx_los"]      = index_episode.idx_los.fillna(0)
index_episode["idx_ed"]       = index_episode.idx_ed.fillna(0).astype(int)
print("admission beginning on the index date (%), at-index cohort:",
      round(100 * index_episode[index_episode.subject_id.isin(at.subject_id)].has_idx_adm.mean(), 1))

## 6 · Follow-up and the landmark samples

Follow-up ends at the earliest of: end of database coverage, day 1,460, 365 days after the last hospital or ED contact (end of death capture), or death. The same end applies to hospitalization and death. Outcomes are counted from day 730: the first cardiovascular hospitalization after day 730 is the event; death without one is the competing event.

In [ ]:
o = oc.assign(mace_date=pd.to_datetime(oc.mace_date), death_date=pd.to_datetime(oc.death_date),
              index_date=pd.to_datetime(oc.index_date))
o["mace_day"]  = (o.mace_date  - o.index_date).dt.days
o["death_day"] = (o.death_date - o.index_date).dt.days

d = (at.merge(o[["subject_id", "prior_mace", "mace_day", "death_day"]], on="subject_id", how="left")
       .merge(gaps, on="subject_id", how="left")
       .merge(index_episode, on="subject_id", how="left"))

# follow-up end; a patient with no hospital/ED contact gets a capture end far below day 730
d["fu_end"] = np.minimum.reduce([d.admin_day.astype(float).values,
                                 np.full(len(d), float(FOLLOWUP_END_DAY)),
                                 d.capture_end_day.fillna(-1e6).values])
dead = d.death_day.notna() & (d.death_day <= d.fu_end)
d.loc[dead, "fu_end"] = d.loc[dead, "death_day"]

def landmark_outcome(F):
    ev  = F.mace_day.notna() & (F.mace_day > LANDMARK_DAY) & (F.mace_day <= F.fu_end)
    dth = ~ev & F.death_day.notna() & (F.death_day > LANDMARK_DAY) & (F.death_day <= F.fu_end)
    status = np.select([ev, dth], [1, 2], 0)
    time = np.where(ev, F.mace_day, np.where(dth, F.death_day, F.fu_end)) - LANDMARK_DAY
    return status, time

d["status"], d["time"] = landmark_outcome(d)

no_event_1_730 = ~d.mace_day.between(1, LANDMARK_DAY)
no_prior       = d.prior_mace.fillna(0) == 0
fu_after       = d.fu_end > LANDMARK_DAY
alive_later    = d.status730 == "alive: record after day 730"

d["in_landmark"]          = alive_later & no_event_1_730 & no_prior & fu_after
d["in_landmark_prior_ok"] = alive_later & no_event_1_730 & fu_after
no_death730 = d.days_to_death.isna() | (d.days_to_death > LANDMARK_DAY)
d["in_day730_info"]       = no_death730 & (d.hosp_366_730.fillna(0) == 1) & no_event_1_730 & no_prior & fu_after

### Figure 1A flow (association branch)

In [ ]:
flow = [("At-index cohort", len(d))]
keep = pd.Series(True, index=d.index)
for label, mask in [("Exclude: died by day 730",                    d.status730 != "died by day 730"),
                    ("Exclude: survival at day 730 not confirmed",  d.status730 != "survival unknown"),
                    ("Exclude: alive at day 730, no later record",  alive_later),
                    ("Exclude: CV hospitalization before index or days 1-730", no_event_1_730 & no_prior),
                    ("Exclude: no follow-up after day 730",         fu_after)]:
    before = int(keep.sum()); keep &= mask
    flow.append((f"{label} ({before - int(keep.sum()):,})", int(keep.sum())))
for s, n in flow:
    print(f"{n:>8,}  {s}")
assert keep.equals(d.in_landmark)

### Checks against the paper

With `REPLICATE_OCTOBER = True`, every line should say `OK`. With `False`, the differences are the size of the correction.

In [ ]:
lm = d[d.in_landmark]
v2 = d[d.in_day730_info]
pk = d[d.in_landmark_prior_ok]

print("PRIMARY LANDMARK POPULATION")
check("  patients",                         len(lm), 14776)
check("  cardiovascular hospitalizations",  int((lm.status == 1).sum()), 645)
check("  deaths without one",               int((lm.status == 2).sum()), 1023)
early = lm[(lm.status == 0) & (lm.time < 730)]
check("  censored before 730 days",         len(early), 3410)
check("    at end of death capture",        int((early.fu_end != early.admin_day).sum()), 2909)
check("    at end of database coverage",    int((early.fu_end == early.admin_day).sum()), 501)
print(f"    median days to early censoring: {early.time.median():.0f} (paper: 439)")

print("\nGAP IN RECORDED CONTACT (landmark population)")
check("  gap > 365 days",                   int(lm.gapped.sum()), 3374)
check("  events with gap",                  int(((lm.status == 1) & (lm.gapped == 1)).sum()), 130)
check("  deaths with gap",                  int(((lm.status == 2) & (lm.gapped == 1)).sum()), 211)
q = lm.longest_gap.quantile([.25, .5, .75]).round(0)
print(f"    longest interval, median (IQR): {q[.5]:.0f} ({q[.25]:.0f}-{q[.75]:.0f})   paper: 202 (133-352)")
k = lm.longest_gap.quantile([.05, .35, .65, .95]).round(0).astype(int).tolist()
print(f"    spline knots (5/35/65/95th pct): {k}   paper: [74, 164, 275, 674]")
print(f"    admission beginning on index date: {100 * lm.has_idx_adm.mean():.1f}%   paper: 7.5%")

print("\nDAY-730-INFORMATION SAMPLE")
check("  patients",                         len(v2), 7374)
check("  cardiovascular hospitalizations",  int((v2.status == 1).sum()), 312)
check("  also in primary population",       int((d.in_day730_info & d.in_landmark).sum()), 6312)

print("\nEVENTS BEFORE INDEX KEPT")
check("  patients",                         len(pk), 15962)
check("  cardiovascular hospitalizations",  int((pk.status == 1).sum()), 775)

print("\nOLD ELIGIBILITY RULES")
n_fail = int((~(lm.bp_rule & lm.enc_rule)).sum())
print(f"  landmark patients who fail the >=2 BP / >=2 encounter rules: {n_fail:,}"
      f"   (paper: 'no patient affected by either rule reached the landmark population')")

## 7 · Save

In [ ]:
suffix = "" if not REPLICATE_OCTOBER else "_october"
save_df(gaps,          DATA_DIR, "gaps")
save_df(quarterly,     DATA_DIR, "quarterly_unique")
save_df(index_episode, DATA_DIR, "index_episode")
save_df(d,             DATA_DIR, f"landmark_build{suffix}")

save_json({
    "mode": MODE,
    "figure1a_flow": [{"step": s, "n": n} for s, n in flow],
    "primary_landmark": {"n": len(lm), "events": int((lm.status == 1).sum()),
                         "deaths": int((lm.status == 2).sum()), "censored_before_730": len(early)},
    "day730_information_sample": {"n": len(v2), "events": int((v2.status == 1).sum())},
    "events_before_index_kept": {"n": len(pk), "events": int((pk.status == 1).sum())},
    "gap_gt_365": {"n": int(lm.gapped.sum()),
                   "events": int(((lm.status == 1) & (lm.gapped == 1)).sum()),
                   "deaths": int(((lm.status == 2) & (lm.gapped == 1)).sum())},
    "gap_spline_knots": k,
    "landmark_patients_failing_old_rules": n_fail,
}, RESULTS_DIR, f"02_landmark_counts_{MODE}")